In [ ]:
!pip install rdkit-pypi
!pip install mordred

In [2]:
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

import pandas as pd
import numpy as np
from mordred import Calculator, descriptors

In [3]:
url = 'https://raw.githubusercontent.com/gashawmg/molecular-descriptors/main/Orbital_Energies_input_data.csv'
dataset = pd.read_csv(url)
dataset.shape

(2904, 2)

In [4]:
dataset.head()

,SMILES,Energygap
0,Cc1ccc(cc1)C(F)(F)F,197.749421
1,OC(=O)CCCCl,247.493942
2,CC(C)(Oc1ccc(CCNC(=O)c2ccc(Cl)cc2)cc1)C(=O)O,164.712327
3,Nc1ccc(Cl)c(Cl)c1,169.027707
4,C[C@@H](CCO)CCC=C(C)C,209.569808


## 1. Generate canonical SMILES

In [5]:
from rdkit import Chem

# 同一个分子（苯）的不同SMILES表示
smiles_list = [
    'c1ccccc1',           # 方式1：环形结构
    'c1ccccc1',           # 方式2：同上
    'c1ccccc1',           # 方式3：同上
    'C1=CC=CC=C1',        # 方式4：使用双键表示
    'C1=CC=CC=C1',        # 方式5：同上
    'c1ccccc1',           # 方式6：环形结构
]

# 转换为分子对象再转回SMILES
for i, smi in enumerate(smiles_list, 1):
    mol = Chem.MolFromSmiles(smi)
    canonical_smi = Chem.MolToSmiles(mol)
    print(f"输入 {i}: {smi:15} -> Canonical: {canonical_smi}")

输入 1: c1ccccc1        -> Canonical: c1ccccc1
输入 2: c1ccccc1        -> Canonical: c1ccccc1
输入 3: c1ccccc1        -> Canonical: c1ccccc1
输入 4: C1=CC=CC=C1     -> Canonical: c1ccccc1
输入 5: C1=CC=CC=C1     -> Canonical: c1ccccc1
输入 6: c1ccccc1        -> Canonical: c1ccccc1


In [6]:
from rdkit import Chem

# 示例分子：乙醇
ethanol_variations = [
    'CCO',                    # 标准写法
    'OCC',                    # 原子顺序不同
    'C(C)O',                  # 括号表示分支
    'C-C-O',                  # 使用单键连接
    '[CH3][CH2][OH]',         # 显式氢原子
    'C(O)C',                  # 另一种分支表示
]

print("=== Canonical SMILES 示例 ===")
print("分子：乙醇\n")

for i, smi in enumerate(ethanol_variations, 1):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        canonical = Chem.MolToSmiles(mol)
        print(f"变体 {i}: {smi:15} -> Canonical: {canonical}")

=== Canonical SMILES 示例 ===
分子：乙醇

变体 1: CCO             -> Canonical: CCO
变体 2: OCC             -> Canonical: CCO
变体 3: C(C)O           -> Canonical: CCO
变体 4: C-C-O           -> Canonical: CCO
变体 5: [CH3][CH2][OH]  -> Canonical: CCO
变体 6: C(O)C           -> Canonical: CCO


In [7]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

def demonstrate_canonical_features():
    """展示 Canonical SMILES 的主要特点"""
    
    print("=== Canonical SMILES 的主要特点 ===\n")
    
    # 1. 原子顺序标准化
    print("1. 原子顺序标准化：")
    mol1 = Chem.MolFromSmiles('CCO')    # 乙醇
    mol2 = Chem.MolFromSmiles('OCC')    # 相同分子，不同顺序
    
    print(f"   CCO  -> {Chem.MolToSmiles(mol1)}")
    print(f"   OCC  -> {Chem.MolToSmiles(mol2)}")
    
    # 2. 芳香性标准化
    print("\n2. 芳香性标准化：")
    benzene_aromatic = Chem.MolFromSmiles('c1ccccc1')    # 芳香表示
    benzene_kekule = Chem.MolFromSmiles('C1=CC=CC=C1')   # 凯库勒表示
    
    print(f"   c1ccccc1    -> {Chem.MolToSmiles(benzene_aromatic)}")
    print(f"   C1=CC=CC=C1 -> {Chem.MolToSmiles(benzene_kekule)}")
    
    # 3. 分支处理标准化
    print("\n3. 分支处理标准化：")
    isopropanol_1 = Chem.MolFromSmiles('CC(C)O')    # 异丙醇
    isopropanol_2 = Chem.MolFromSmiles('OC(C)C')    # 不同顺序
    
    print(f"   CC(C)O -> {Chem.MolToSmiles(isopropanol_1)}")
    print(f"   OC(C)C -> {Chem.MolToSmiles(isopropanol_2)}")
    
    # 4. 立体化学处理
    print("\n4. 立体化学处理：")
    alanine_l = Chem.MolFromSmiles('C[C@H](N)C(=O)O')   # L-丙氨酸
    alanine_d = Chem.MolFromSmiles('C[C@@H](N)C(=O)O')  # D-丙氨酸
    
    print(f"   L-Ala: {Chem.MolToSmiles(alanine_l)}")
    print(f"   D-Ala: {Chem.MolToSmiles(alanine_d)}")

demonstrate_canonical_features()

=== Canonical SMILES 的主要特点 ===

1. 原子顺序标准化：
   CCO  -> CCO
   OCC  -> CCO

2. 芳香性标准化：
   c1ccccc1    -> c1ccccc1
   C1=CC=CC=C1 -> c1ccccc1

3. 分支处理标准化：
   CC(C)O -> CC(C)O
   OC(C)C -> CC(C)O

4. 立体化学处理：
   L-Ala: C[C@H](N)C(=O)O
   D-Ala: C[C@@H](N)C(=O)O


In [8]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
from collections import Counter

def canonical_smiles_applications():
    """展示 Canonical SMILES 的实际应用"""
    
    # 创建一个包含重复分子的数据集
    molecules = [
        ('Aspirin', 'CC(=O)OC1=CC=CC=C1C(=O)O'),
        ('Aspirin', 'OC(=O)C1=CC=CC=C1OC(C)=O'),  # 相同分子，不同顺序
        ('Aspirin', 'c1ccccc1OC(=O)C(=O)O'),       # 错误写法
        ('Caffeine', 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'),
        ('Caffeine', 'Cn1c(=O)c2c(ncn2C)n(C1=O)C'), # 相同分子，不同顺序
        ('Ethanol', 'CCO'),
        ('Ethanol', 'OCC'),                         # 相同分子，不同顺序
        ('Benzene', 'c1ccccc1'),
        ('Benzene', 'C1=CC=CC=C1'),                  # 相同分子，不同顺序
    ]
    
    print("=== Canonical SMILES 应用示例 ===\n")
    
    # 1. 去重处理
    print("1. 使用 Canonical SMILES 去重：")
    unique_molecules = {}
    
    for name, smi in molecules:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            canonical = Chem.MolToSmiles(mol)
            if canonical not in unique_molecules:
                unique_molecules[canonical] = name
    
    print(f"   原始数据: {len(molecules)} 条记录")
    print(f"   去重后: {len(unique_molecules)} 个唯一分子")
    print("   唯一分子:")
    for canonical, name in unique_molecules.items():
        print(f"     - {name}: {canonical}")
    
    # 2. 分子数据库索引
    print("\n2. 作为数据库索引键：")
    mol_db = {}
    for canonical, name in unique_molecules.items():
        mol = Chem.MolFromSmiles(canonical)
        mol_db[canonical] = {
            'name': name,
            'mw': Descriptors.MolWt(mol),
            'logP': Descriptors.MolLogP(mol),
            'hba': Descriptors.NumHAcceptors(mol),
            'hbd': Descriptors.NumHDonors(mol),
        }
    
    df = pd.DataFrame(mol_db).T
    print(df)
    
    # 3. 分子指纹生成
    print("\n3. 确保指纹一致性：")
    from rdkit.Chem import AllChem
    
    print("   对同一分子的不同表示生成指纹：")
    aspirin_variations = [
        'CC(=O)OC1=CC=CC=C1C(=O)O',
        'OC(=O)C1=CC=CC=C1OC(C)=O',
        'c1ccccc1OC(=O)C(=O)O',
    ]
    
    for i, smi in enumerate(aspirin_variations, 1):
        mol = Chem.MolFromSmiles(smi)
        canonical = Chem.MolToSmiles(mol)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        fp_str = fp.ToBitString()[:20] + "..."  # 只显示前20位
        
        print(f"   变体 {i}:")
        print(f"     原始SMILES: {smi}")
        print(f"     Canonical: {canonical}")
        print(f"     指纹(前20位): {fp_str}")
        print()

canonical_smiles_applications()

=== Canonical SMILES 应用示例 ===

1. 使用 Canonical SMILES 去重：
   原始数据: 9 条记录
   去重后: 5 个唯一分子
   唯一分子:
     - Aspirin: CC(=O)Oc1ccccc1C(=O)O
     - Aspirin: O=C(O)C(=O)Oc1ccccc1
     - Caffeine: Cn1c(=O)c2c(ncn2C)n(C)c1=O
     - Ethanol: CCO
     - Benzene: c1ccccc1

2. 作为数据库索引键：
                                name       mw    logP hba hbd
CC(=O)Oc1ccccc1C(=O)O        Aspirin  180.159  1.3101   3   1
O=C(O)C(=O)Oc1ccccc1         Aspirin  166.132  0.6766   3   1
Cn1c(=O)c2c(ncn2C)n(C)c1=O  Caffeine  194.194 -1.0293   6   0
CCO                          Ethanol   46.069 -0.0014   1   1
c1ccccc1                     Benzene   78.114  1.6866   0   0

3. 确保指纹一致性：
   对同一分子的不同表示生成指纹：
   变体 1:
     原始SMILES: CC(=O)OC1=CC=CC=C1C(=O)O
     Canonical: CC(=O)Oc1ccccc1C(=O)O
     指纹(前20位): 00000000000100000000...

   变体 2:
     原始SMILES: OC(=O)C1=CC=CC=C1OC(C)=O
     Canonical: CC(=O)Oc1ccccc1C(=O)O
     指纹(前20位): 00000000000100000000...

   变体 3:
     原始SMILES: c1ccccc1OC(=O)C(=O)O
     Canonical: O=C(O

[18:42:39] DEPRECATION WARNING: please use MorganGenerator
[18:42:39] DEPRECATION WARNING: please use MorganGenerator
[18:42:39] DEPRECATION WARNING: please use MorganGenerator


In [11]:
def canonical_smiles(smiles):
    mols = [Chem.MolFromSmiles(smi) for smi in smiles] 
    smiles = [Chem.MolToSmiles(mol) for mol in mols]
    return smiles

In [12]:
# Canonical SMILES
Canon_SMILES = canonical_smiles(dataset.SMILES)
len(Canon_SMILES)

2904

In [13]:
# Put the smiles in the dataframe
dataset['SMILES'] = Canon_SMILES
dataset

,SMILES,Energygap
0,Cc1ccc(C(F)(F)F)cc1,197.749421
1,O=C(O)CCCCl,247.493942
2,CC(C)(Oc1ccc(CCNC(=O)c2ccc(Cl)cc2)cc1)C(=O)O,164.712327
3,Nc1ccc(Cl)c(Cl)c1,169.027707
4,CC(C)=CCC[C@@H](C)CCO,209.569808
...,...,...
2899,c1ccc(P(CCP(c2ccccc2)c2ccccc2)c2ccccc2)cc1,168.649319
2900,Brc1cccc2sccc12,162.928319
2901,CCOC(=O)N1c2ccccc2C=C[C@@H]1OCC,165.098245
2902,c1ccc2sccc2c1,167.958431


## 2.  Drop duplicate values

### a. General molecular descriptors-about 200 molecular descriptors

In [14]:
dataset_new = dataset.drop_duplicates(subset=['SMILES'])
len(dataset_new)

2873

In [15]:
dataset_new

,SMILES,Energygap
0,Cc1ccc(C(F)(F)F)cc1,197.749421
1,O=C(O)CCCCl,247.493942
2,CC(C)(Oc1ccc(CCNC(=O)c2ccc(Cl)cc2)cc1)C(=O)O,164.712327
3,Nc1ccc(Cl)c(Cl)c1,169.027707
4,CC(C)=CCC[C@@H](C)CCO,209.569808
...,...,...
2899,c1ccc(P(CCP(c2ccccc2)c2ccccc2)c2ccccc2)cc1,168.649319
2900,Brc1cccc2sccc12,162.928319
2901,CCOC(=O)N1c2ccccc2C=C[C@@H]1OCC,165.098245
2902,c1ccc2sccc2c1,167.958431


In [16]:
def RDkit_descriptors(smiles):
    mols = [Chem.MolFromSmiles(i) for i in smiles] 
    calc = MoleculeDescriptors.MolecularDescriptorCalculator([x[0] for x in Descriptors._descList])
    desc_names = calc.GetDescriptorNames()
    
    Mol_descriptors =[]
    for mol in mols:
        # add hydrogens to molecules
        mol=Chem.AddHs(mol)
        # Calculate all 200 descriptors for each molecule
        descriptors = calc.CalcDescriptors(mol)
        Mol_descriptors.append(descriptors)
    return Mol_descriptors,desc_names 

# Function call
Mol_descriptors,desc_names = RDkit_descriptors(dataset_new['SMILES'])

In [17]:
df_with_200_descriptors = pd.DataFrame(Mol_descriptors,columns=desc_names)
df_with_200_descriptors

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,12.550510,12.550510,1.008796,-5.076351,0.546828,21.909091,160.138,153.082,160.049985,60,...,0,0,0,0,0,0,0,0,0,0
1,10.676844,10.676844,1.840718,-3.333333,0.569323,29.000000,122.551,115.495,122.013457,42,...,0,0,0,0,0,0,0,0,0,0
2,13.050084,13.050084,0.722809,-4.111425,0.790287,24.520000,361.825,341.665,361.108086,132,...,0,0,0,0,0,0,0,0,0,0
3,7.402685,7.402685,0.074321,-0.449630,0.582519,16.888889,162.019,156.979,160.979905,48,...,0,0,0,0,0,0,0,0,0,0
4,8.095237,8.095237,1.886963,-4.484184,0.606746,50.909091,156.269,136.109,156.151415,66,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2868,9.505488,9.505488,0.973292,-3.873136,0.373065,24.214286,398.426,374.234,398.135324,138,...,0,0,0,0,0,0,0,0,0,0
2869,7.651157,7.651157,0.017477,-0.170718,0.625891,17.500000,213.099,208.059,211.929533,50,...,0,0,0,0,0,0,0,1,0,0
2870,13.017078,13.017078,0.325694,-3.813937,0.823664,39.333333,247.294,230.158,247.120843,96,...,0,0,0,0,0,0,0,0,0,0
2871,7.592407,7.592407,0.030556,-0.348333,0.519376,19.555556,134.203,128.155,134.019021,44,...,0,0,0,0,0,0,0,1,0,0


### b. Fingerprints

In [19]:
def morgan_fpts(data):
    Morgan_fpts = []
    for i in data:
        mol = Chem.MolFromSmiles(i) 
        fpts =  AllChem.GetMorganFingerprintAsBitVect(mol,2,2048)
        mfpts = np.array(fpts)
        Morgan_fpts.append(mfpts)  
    return np.array(Morgan_fpts)

In [ ]:
Morgan_fpts = morgan_fpts(dataset_new['SMILES'])
Morgan_fpts.shape

In [21]:
Morgan_fingerprints = pd.DataFrame(Morgan_fpts,columns=['Col_{}'.format(i) for i in range(Morgan_fpts.shape[1])])
Morgan_fingerprints

,Col_0,Col_1,Col_2,Col_3,Col_4,Col_5,Col_6,Col_7,Col_8,Col_9,...,Col_2038,Col_2039,Col_2040,Col_2041,Col_2042,Col_2043,Col_2044,Col_2045,Col_2046,Col_2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2868,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2869,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2870,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2871,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Calculate descreptors using Mordred-1826 descriptors

In [22]:
def All_Mordred_descriptors(data):
    calc = Calculator(descriptors, ignore_3D=False)
    mols = [Chem.MolFromSmiles(smi) for smi in data]
    
    # pandas df
    df = calc.pandas(mols)
    return df

In [23]:
mordred_descriptors = All_Mordred_descriptors(dataset_new['SMILES'])

100%|██████████| 2873/2873 [01:35<00:00, 29.97it/s]


In [24]:
mordred_descriptors

,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,12.527341,2.311476,4.622953,12.527341,1.138849,3.302522,...,9.182249,41.326257,160.049985,8.891666,152,13,54.0,59.0,5.284722,2.333333
1,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,1,0,7.727407,1.931852,3.863703,7.727407,1.103915,2.752227,...,7.321850,31.336140,122.013457,8.715247,52,4,24.0,22.0,3.861111,1.833333
2,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,1,0,30.648742,2.324224,4.648448,30.648742,1.225950,4.118873,...,9.931735,59.295845,361.108086,8.024624,1882,35,124.0,139.0,9.729167,5.486111
3,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,10.792280,2.245827,4.491654,10.792280,1.199142,3.099448,...,8.806724,37.839725,160.979905,11.498565,84,10,42.0,46.0,4.083333,2.027778
4,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,12.133645,2.047810,4.095621,12.133645,1.103059,3.219224,...,8.131825,38.565088,156.151415,5.037142,194,9,42.0,41.0,5.472222,2.750000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2868,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,38.253985,2.374938,4.749876,38.253985,1.366214,4.258939,...,10.073357,62.971588,398.135324,7.656449,2014,41,142.0,164.0,6.166667,6.361111
2869,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,13.098358,2.369838,4.633950,13.098358,1.309836,3.261311,...,9.161465,53.745115,211.929533,14.128636,105,12,52.0,61.0,2.833333,2.222222
2870,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,22.978744,2.442763,4.885526,22.978744,1.276597,3.800055,...,9.763593,50.871918,247.120843,7.060596,574,28,88.0,103.0,6.055556,4.277778
2871,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,12.170709,2.322596,4.516123,12.170709,1.352301,3.160409,...,8.914761,51.887188,134.019021,8.934601,79,9,46.0,53.0,1.972222,2.027778
